# DuoFusion: learned extrapolation of truncated CTP curves (baseline)

Learned baseline of the truncation study. A 1D network sees the first L seconds of a baseline-subtracted
CTP curve (L random in 20-45 s) and predicts the curve up to 70 s. Five models are trained, one per
patient fold of the evaluation (fold k is never seen by model k). The weights go to
`MyDrive/duofusion/learned_extrap/`; scripts/osvd_volumes.py uses them to extend the tissue curves and the AIF before sSVD,
exactly like the gamma-variate baseline.

Runtime: GPU (H100 or RTX 6000), High-RAM. Order: run all cells. Download ~27 GB, curve extraction
~20-30 min, training 5 folds ~10-20 min on an H100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install nibabel pyarrow huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/duofusion/learned_extrap'
os.makedirs(OUT, exist_ok=True)

In [ ]:
from huggingface_hub import snapshot_download
HF = '/content/isles24_hf'
snapshot_download('hugging-science/isles24-stroke', repo_type='dataset', local_dir=HF,
                  revision='9707a7fca6d3dd1a690de010ec4aed06bdcd0417',   # the revision used in the paper
                  allow_patterns=['data/*.parquet'], max_workers=16)
print(len(os.listdir(f'{HF}/data')), 'files')

In [ ]:
FOLDS = {"1": 2, "2": 2, "3": 3, "4": 0, "5": 4, "6": 0, "7": 3, "8": 1, "9": 3, "10": 3, "11": 0, "12": 2, "13": 4, "14": 3, "15": 1, "16": 2, "17": 1, "19": 3, "20": 0, "21": 0, "22": 0, "25": 2, "26": 0, "27": 3, "28": 0, "30": 3, "33": 2, "36": 3, "37": 1, "38": 2, "40": 1, "43": 4, "45": 0, "47": 4, "48": 3, "49": 0, "52": 1, "53": 3, "54": 2, "55": 2, "57": 4, "62": 0, "66": 2, "68": 4, "70": 3, "71": 4, "73": 4, "74": 2, "75": 4, "76": 4, "77": 1, "78": 4, "79": 3, "80": 1, "81": 4, "82": 1, "83": 4, "84": 2, "85": 4, "86": 3, "87": 0, "88": 1, "89": 0, "90": 2, "91": 2, "92": 3, "93": 1, "94": 2, "95": 1, "96": 4, "97": 2, "98": 1, "99": 3, "100": 0, "101": 1, "102": 1, "103": 4, "104": 0, "105": 2, "106": 3, "107": 1, "108": 1, "109": 2, "110": 0, "111": 3, "112": 4, "113": 2, "114": 4, "115": 4, "116": 0, "117": 3, "118": 2, "119": 4, "133": 0, "134": 4, "135": 2, "136": 2, "137": 1, "138": 1, "139": 3, "140": 2, "141": 0, "142": 3, "143": 1, "144": 3, "145": 0, "146": 3, "147": 1, "148": 0, "149": 4, "150": 0, "151": 1, "152": 1, "153": 1, "154": 4, "155": 2, "156": 3, "157": 0, "158": 0, "159": 1, "161": 2, "162": 4, "163": 1, "164": 2, "165": 0, "166": 3, "167": 4, "168": 0, "169": 3, "170": 2, "171": 2, "172": 1, "173": 2, "174": 4, "175": 4, "176": 0, "177": 1, "178": 4, "179": 3, "180": 3, "181": 1, "182": 0, "183": 3, "184": 1, "185": 0, "186": 0, "187": 3, "188": 2, "189": 4}
FOLDS = {int(k): v for k, v in FOLDS.items()}
TMAX = 70          # model output length (frames at 1 s)
PER_PATIENT = 20000
ARTERIAL = 1000    # high-peak voxels per patient so the model also sees AIF-like curves

In [ ]:
import glob, io, tempfile, numpy as np, nibabel as nib, pyarrow.parquet as pq
from concurrent.futures import ProcessPoolExecutor

def baseline_frames(ctp, brain):
    m = ctp[brain].mean(axis=0)
    rise = m - m[:3].mean()
    return max(2, int(np.argmax(rise > 0.1 * rise.max())) - 1)

def extract(path):
    r = pq.read_table(path).to_pylist()[0]
    num = int(r['subject_id'].split('stroke')[1])
    if num not in FOLDS:
        return None
    with tempfile.NamedTemporaryFile(suffix='.nii.gz') as f:
        f.write(r['ctp']['bytes']); f.flush()
        ctp = np.asarray(nib.load(f.name).get_fdata(), np.float32)
    fov = np.all(ctp > -1000, axis=-1)
    mean = ctp.mean(-1)
    brain = fov & (mean > 0) & (mean < 100)
    nb = baseline_frames(ctp, brain)
    ctc = ctp[brain] - ctp[brain][:, :nb].mean(-1, keepdims=True)
    T = min(ctc.shape[1], TMAX)
    ctc = ctc[:, :T]
    peak = ctc.max(1)
    rng = np.random.default_rng(num)
    ok = np.nonzero(peak > 5)[0]
    tis = rng.choice(ok, min(PER_PATIENT, ok.size), replace=False)
    art = np.argsort(-peak)[:ARTERIAL]
    sel = np.concatenate([tis, art])
    cur = np.zeros((sel.size, TMAX), np.float16)
    cur[:, :T] = ctc[sel]
    return num, cur, T

files = sorted(glob.glob(f'{HF}/data/train-*.parquet'))
curves, pid, length = [], [], []
with ProcessPoolExecutor(max_workers=8) as ex:
    for res in ex.map(extract, files):
        if res is None: continue
        num, cur, T = res
        curves.append(cur); pid.append(np.full(len(cur), num)); length.append(np.full(len(cur), T))
curves, pid, length = np.concatenate(curves), np.concatenate(pid), np.concatenate(length)
np.savez_compressed(f'{OUT}/curves_isles24.npz', curves=curves, pid=pid, length=length)
print(curves.shape, len(set(pid)), 'patients; lengths', np.unique(length))

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
dev = 'cuda'

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.c1 = nn.Conv1d(c, c, 3, padding=d, dilation=d)
        self.c2 = nn.Conv1d(c, c, 3, padding=d, dilation=d)
        self.n = nn.GroupNorm(8, c)
    def forward(self, x):
        return x + self.c2(F.gelu(self.n(self.c1(F.gelu(x)))))

class Extrap(nn.Module):
    """Input (B, 3, TMAX): observed curve / its observed max (zeros after L), observation mask,
    time. Output: normalised curve over TMAX; observed frames are kept as measured."""
    def __init__(self, c=96):
        super().__init__()
        self.inp = nn.Conv1d(3, c, 1)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16, 1, 2, 4, 8, 16)])
        self.glob = nn.Sequential(nn.Linear(c * TMAX, 256), nn.GELU(), nn.Linear(256, TMAX))
        self.out = nn.Conv1d(c, 1, 1)
    def forward(self, x):
        h = self.blocks(self.inp(x))
        y = self.out(h)[:, 0] + self.glob(h.flatten(1))
        return x[:, 1] * x[:, 0] + (1 - x[:, 1]) * y

def make_batch(cur, T, Lmin=20, Lmax=45, gen=None):
    B = cur.shape[0]
    L = torch.randint(Lmin, Lmax + 1, (B,), device=dev, generator=gen)
    L = torch.minimum(L, T - 5)
    t = torch.arange(TMAX, device=dev)[None, :]
    obs = (t < L[:, None]).float()
    scale = (cur * obs).amax(1).clamp(min=1.0)
    x = torch.stack([cur * obs / scale[:, None], obs, t.expand(B, -1).float() / TMAX], 1)
    target = cur / scale[:, None]
    loss_mask = ((t >= L[:, None]) & (t < T[:, None])).float()
    return x, target, loss_mask

def train_fold(k, epochs=6, bs=8192, lr=2e-3):
    tr = np.array([FOLDS[p] != k for p in pid])
    cur = torch.tensor(curves[tr], dtype=torch.float32, device=dev)
    T = torch.tensor(length[tr], device=dev)
    model = Extrap().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * (len(cur) // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    model.train()
    for s in range(steps):
        i = torch.randint(0, len(cur), (bs,), device=dev)
        x, y, m = make_batch(cur[i], T[i])
        with torch.autocast('cuda', dtype=torch.bfloat16):
            pred = model(x)
        loss = ((pred.float() - y) ** 2 * m).sum() / m.sum().clamp(min=1)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if s % 500 == 0:
            print(f'fold {k} step {s}/{steps} loss {loss.item():.4f}')
    return model

@torch.no_grad()
def tail_error(model, k, L):
    te = np.array([FOLDS[p] == k for p in pid])
    cur = torch.tensor(curves[te], dtype=torch.float32, device=dev)
    T = torch.tensor(length[te], device=dev)
    g = torch.Generator(device=dev).manual_seed(0)
    x, y, m = make_batch(cur, T, L, L, g)
    model.eval()
    pred = model(x).float()
    return float(((pred - y).abs() * m).sum() / m.sum())

In [ ]:
for k in range(5):
    model = train_fold(k)
    torch.save(model.state_dict(), f'{OUT}/extrap_fold{k}.pt')
    print('fold', k, 'held-out tail MAE (normalised) at L=25/30/35/40:',
          [round(tail_error(model, k, L), 4) for L in (25, 30, 35, 40)])
!ls -la {OUT}

Done: copy the five `extrap_fold*.pt` files (they are in Google Drive) and point `--models` of scripts/osvd_volumes.py to them.